# Modeling

Packages and setup

In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 
os.chdir(PROJECT_ROOT)
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6, DATA_DIR_3_7, _ = DATA_DIR_3_x
from foodcast.tools.rolling import rolling_window_avg, add_interday_variables, add_intraday_variables, unroll, season_from_month
from foodcast.tools.takeout import takeout

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
data = load_all_res_3_4_ai()

# Load special data
customers_before_after = pd.read_pickle(DATA_DIR_3 / 'customers_before_after.pkl')
grouping_mappings = pd.read_pickle(DATA_DIR_3 / 'grouping_mappings.pkl')
animal_categories_items = pd.read_pickle(DATA_DIR_3 / 'animal_categories_items.pkl')

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

    animal_product_categories = [
        'lamb','chunked_beef_or_pork','pulled_pork', # A_1
        'beef_or_pork_burger','ground_meat','meatballs', # A_2
        'sausage','bacon','breakfast_sausage_patty', # A_3
        'unfried_chicken','fried_chicken', # A_4
        'savory_dairy','sweet_dairy', # A_5
        'egg' # No MPBAs
    ]

    category_mappings = {
        'lamb': ('|'.join(['lamb','sheep','mutton']),
                 '|'.join(['-v','v-','^v ',' v$','vegan','black sheep'])),
        'beef_or_pork_burger': ('|'.join(['burg','patty']),
                                '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond'])),
        'sausage': ('|'.join(['sausage','brat','link','kielbasa']),
                    '|'.join(['-v','v-','^v ',' v$','vegan','beyond','impossible'])),
        'meatballs': ('|'.join(['meatball','kofta']),
                      '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond'])),
        'bacon': ('|'.join(['bacon','pancetta','prosciutto']),
                  '|'.join(['-v','v-','^v ',' v$','vegan','thrilling','bakon','vegan bacon'])),
        'ground_meat': ('|'.join(['ground pork','ground beef', 'ground meat', 'ground chicken', 'ground lamb','mince']),
                        '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond','tofu','seitan'])),
        'breakfast_sausage_patty': ('|'.join(['sausage patty','breakfast sausage','breakfast patty']),
                                    '|'.join(['-v','v-','^v ',' v$','vegan','impossible','beyond'])),
        'chunked_beef_or_pork': ('|'.join(['beef chunks','carnitas']),
                                 '|'.join(['-v','v-','^v ',' v$','vegan','tofu','seitan'])),
        'pulled_pork': ('|'.join(['pulled','pork']),
                        '|'.join(['-v','v-','^v ',' v$','vegan','jackfruit'])),
        'unfried_chicken': ('|'.join(['chicken','turkey']),
                            '|'.join(['-v','v-','^v ',' v$','vegan','unchicken','un\'chicken'])),
        'fried_chicken': ('|'.join(['fried chicken','wing','tender','nugget','drumstick','popcorn chicken']),
                          '|'.join(['-v','v-','^v ',' v$','vegan','tofu','seitan'])),
        'savory_dairy': ('|'.join(['cheese','chz','cream','yogurt','aioli',
                                   'butter','feta','mozzarella','cheddar',
                                   'parmesan','queso','ricotta']),
                         '|'.join(['-v','v-','^v ',' v$','vegan','dairy-free','df','non-dairy','dairy free'])),
        'sweet_dairy': ('|'.join(['dessert','cake','custard','cream','cheescake','pudding',]),
                        '|'.join(['-v','v-','^v ',' v$','vegan','dairy-free','df','non-dairy','dairy free'])),
        'egg': ('|'.join(['egg','mayo','aioli','omelet','omelette','scramble','deviled']),
                '|'.join(['-v','v-','^v ',' v$','vegan','just egg']))
    }

    def targeted_from_map(df, mapping, default=0, dtype="int8"):
        """
        mapping: list of (location_id_list, source_colname)
        Returns a Series that, for each row, uses the source column specified by
        the first matching location_id_list in mapping; otherwise default.
        """
        out = pd.Series(default, index=df.index)
        for locs, col in mapping:
            mask = df["location_id"].isin(locs)
            out.loc[mask] = df.loc[mask, col].astype(dtype).values
        return out.astype(dtype)

    totals = [0, 0, 0]
    for loc_id in location_ids_by_coverage:
        totals[0] += data[loc_id].shape[0]

    df_all_list = []   
    for loc_id in location_ids_by_coverage:
        df = (
            data[loc_id]
            .query('~is_drink')
            .query('~is_nonfood')
            .query('~is_nonmeal_merchandise'))

        totals[1] += df.shape[0]
        filepath = Path(DATA_DIR_3_5) / f'{loc_id}.parquet'
        #if not filepath.exists():
        df.to_parquet(filepath, index=False)

        df = (
            df
            .fillna({'item_modifications':''})
            .query('~item_modifications.str.lower().str.contains(@takeout)')
            .query('~item_name.str.lower().str.contains(@takeout)'))
        
        totals[2] += df.shape[0]
        filepath = Path(DATA_DIR_3_6) / f'{loc_id}.parquet'
        #if not filepath.exists():
        df.to_parquet(filepath, index=False)
            
        df = (
            df
            .set_index('created_at')
            .assign(meat = lambda df: ~df.vegetarian)
            .pipe(rolling_window_avg, 'vegan', 'item_price', 'item_quantity', 1, 'D')
            .pipe(rolling_window_avg, 'vegetarian', 'item_price', 'item_quantity', 1, 'D')
            .pipe(rolling_window_avg, 'meat', 'item_price', 'item_quantity', 1, 'D')
            .reset_index()
            .assign(created_at = lambda df: df.created_at.dt.tz_convert('utc')))


        for cat, (keywords, anti_keywords) in category_mappings.items():
            items = animal_categories_items.get((loc_id, cat), [])
            df[cat] = (
                df["item_name"].isin(items)
                | df["item_modifications"].str.contains(keywords, case=False, na=False)
                | df["reasoning_x"].str.contains(keywords, case=False, na=False)
            )
        
        filepath = Path(DATA_DIR_3_7) / f'{loc_id}.parquet'
        #if not filepath.exists():
        df.to_parquet(filepath, index=False)